In [ ]:
import rasterio
import numpy as np
import os

def calculate_forest_depth_statistics_by_year(directory_path, start_year=1985, end_year=2021):
    results = {}

    for year in range(start_year, end_year + 1):
        print(year)
        file_path = os.path.join(directory_path, f'LCMAP_CU_{year}_V13_LCPRI.tif')
        if os.path.exists(file_path):
            with rasterio.open(file_path) as src:
                data = src.read(1)
                # Calculate pixel counts for each category
                pixel_counts = {i: np.sum(data == i) for i in range(1, 6)}
                # Calculate area in square meters
                area = {k: v * (0.03*0.03) for k, v in pixel_counts.items()}
                # Calculate total area to find proportions
                total_area = sum(area.values())
                proportions = {k: v / total_area if total_area > 0 else 0 for k, v in area.items()}

                results[year] = {'area': area, 'proportions': proportions}

    return results

# Usage
directory_path = 'G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Forest_Depth_Classification'
forest_depth_stats_by_year = calculate_forest_depth_statistics_by_year(directory_path)
print(forest_depth_stats_by_year)
# Save the results to CSV
csv_path = os.path.join(directory_path, 'forest_depth_statistics.csv')
forest_depth_stats_by_year.to_csv(csv_path, index=False)

In [ ]:
print(forest_depth_stats_by_year)

In [ ]:
import pandas as pd

df = pd.DataFrame(forest_depth_stats_by_year)

directory_path = 'G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Forest_Depth_Classification'

# Save the results to CSV
csv_path = os.path.join(directory_path, 'forest_depth_statistics.csv')
df.to_csv(csv_path, index=False)

In [ ]:
import os
import pandas as pd
import geopandas as gpd
from tqdm import trange

# Set the folder path
folder_path = r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Depth_Classification\Ecoregion_Classification'

# Iterate through each year from 1985 to 2021
for year in trange(1985, 2022):
    file_name = f"{year}_eco_stats.shp"
    file_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(file_path):
        # Read the shapefile
        gdf = gpd.read_file(file_path)
        
        # Select useful columns
        useful_columns = ['NA_L1NAME', 'Edge_Dep_1', 'Edge_Dep_2', 'Edge_Dep_3', 'Edge_Dep_4','Edge_Dep_5']
        gdf = gdf[useful_columns]
        
        # Group by NA_L1NAME and sum the Edge_cod columns
        grouped_df = gdf.groupby('NA_L1NAME').sum().reset_index()
        grouped_df['Year'] = year  # Add year information
        
        # Save the yearly summary result to a separate file
        output_file_yearly = os.path.join(folder_path, f'eco_stats_summary_{year}.csv')
        grouped_df.to_csv(output_file_yearly, index=False)
    else:
        print(f"File {file_name} does not exist.")

# Set the folder path
folder_path = r'G:\Hangkai\CONUS_Forest_Edge_LCMAP\Forest_Depth_Classification\Ecoregion_Classification'

# Initialize an empty DataFrame to store all years' data
all_data_df = pd.DataFrame()

# Iterate through each year from 1985 to 2021
for year in range(1985, 2022):
    file_name = f'eco_stats_summary_{year}.csv'
    file_path = os.path.join(folder_path, file_name)
    
    if os.path.exists(file_path):
        yearly_df = pd.read_csv(file_path)
        yearly_df['Year'] = year
        all_data_df = pd.concat([all_data_df, yearly_df], ignore_index=True)
    else:
        print(f"File {file_name} does not exist.")

# Save the consolidated data to a CSV file for future use
consolidated_file = os.path.join(folder_path, 'consolidated_eco_stats_1985_2021.csv')
all_data_df.to_csv(consolidated_file, index=False)